# 00 - Configuración de la Entrega 2 (capas, Volumes, tablas y referencias)

**Objetivo.** Crear de forma idempotente el catálogo, los esquemas `bronze`, `silver`, `gold`, `ml`, `ops` (además de `landing` y `reference` de la Entrega 1), los Volumes, las tablas con DDL explícito y las tablas de referencia.

**Entradas.** Widget `catalogo`; WPI de la Entrega 1 en `reference.world_port_index`; shapefiles Natural Earth 1:10m (`ne_10m_land`, `ne_10m_lakes`, `ne_10m_rivers_lake_centerlines`) en `reference.files/natural_earth/` (se descargan si faltan).

**Salidas.** 12 tablas Delta, `reference.{vessel_type_catalog, ports, port_cells_h3r8, land_cells_h3r5}` y `ops.feed_calendar`.

**Validación.** `SHOW TABLES` por esquema y conteos de referencias.

In [0]:
%pip install -q h3==4.5.0 shapely==2.1.2 pyshp==2.3.1 PyYAML==6.0.3

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Si Databricks muestra el mensaje: "you may need to restart the kernel..."
# ejecutar: 
# dbutils.library.restartPython()
# y continuar desde la siguiente celda.

In [0]:
import json
import os
import sys
import urllib.request
from pathlib import Path

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "src")))

from oceanwatch.lakehouse import get_lakehouse
from oceanwatch.notebook_params import widget
from oceanwatch.pipeline import seed_feed_calendar
from oceanwatch.setup import build_reference_tables, create_all

CATALOG = widget(dbutils, "catalogo", "oceanwatch_g06")
lh = get_lakehouse(spark, catalog=CATALOG)

In [0]:
print(json.dumps(create_all(lh), indent=2))

[
  "bronze.ais_raw",
  "silver.ais_posiciones",
  "silver.ais_cuarentena",
  "silver.metricas_calidad",
  "ml.features_velocidad_buque_dia",
  "ops.pipeline_runs",
  "ops.feed_calendar",
  "gold.trayectorias_diarias_buque",
  "gold.estadias_puerto",
  "gold.puertos_resumen_diario",
  "gold.trafico_celda_hora",
  "gold.anomalias_velocidad_diarias"
]


## Referencias geográficas

Puertos: WPI (NGA Pub. 150) conservado en la Entrega 1. Máscara de tierra: Natural Earth 1:10m (dominio público, naciscdn.org).

In [0]:
ne_dir = Path(lh.reference_dir()) / "natural_earth"
ne_dir.mkdir(parents=True, exist_ok=True)
for name, theme in [("ne_10m_land", "physical"), ("ne_10m_lakes", "physical"), ("ne_10m_rivers_lake_centerlines", "physical")]:
    target = ne_dir / f"{name}.zip"
    if not target.exists():
        urllib.request.urlretrieve(f"https://naciscdn.org/naturalearth/10m/{theme}/{name}.zip", target)
    print(name, target.stat().st_size)

wpi_csv = f"/Volumes/{CATALOG}/reference/world_port_index/UpdatedPub150.csv"
print(json.dumps(build_reference_tables(lh, wpi_csv, str(ne_dir)), indent=2))
print("Calendario del feed:", seed_feed_calendar(lh))

ne_10m_land 3269070
ne_10m_lakes 2349685
ne_10m_rivers_lake_centerlines 2079507
{
  "vessel_type_catalog": 34,
  "ports": 3805,
  "port_cells_h3r8": 67922,
  "land_cells_h3r5": 67992
}
Calendario del feed: 14


In [0]:
for schema in ["bronze", "silver", "gold", "ml", "ops", "reference"]:
    display(spark.sql(f"SHOW TABLES IN {CATALOG}.{schema}"))
print("CONFIGURACION_E2_OK")

database,tableName,isTemporary
bronze,ais_raw,false


database,tableName,isTemporary
silver,ais_cuarentena,false
silver,ais_posiciones,false
silver,metricas_calidad,false


database,tableName,isTemporary
gold,anomalias_velocidad_diarias,false
gold,estadias_puerto,false
gold,puertos_resumen_diario,false
gold,trafico_celda_hora,false
gold,trayectorias_diarias_buque,false


database,tableName,isTemporary
ml,features_velocidad_buque_dia,false


database,tableName,isTemporary
ops,feed_calendar,false
ops,pipeline_runs,false


database,tableName,isTemporary
reference,land_cells_h3r5,false
reference,port_cells_h3r8,false
reference,ports,false
reference,vessel_type_catalog,false


CONFIGURACION_E2_OK
